# 02 — MRI pseudo-labels for Vlad and Moe

There are no hand labels or ready-made tissue segmentations for the PS-OCT. Following the lab's suggestion, segment each brain's ex vivo MRI into gray matter, white matter and fluid with ANTs Atropos (`brainseg.mri_labels`), check the labels against the PS-OCT contrasts in MRI space, then warp them into PS-OCT space through the lab's FLIRT + FNIRT registration (`MRI_to_PSOCT_warpfield`).

Files, per subject under `data/<subject>/`:
- `mri/`: `T2map`, `brain_mask` (Vlad) / `nodif_brain_mask` (Moe) from `MRI/exvivo/raw` and `proc/data`; `dti_FA`, `dti_MD`, `dti_V1` from `MRI/exvivo/proc/data_lowershell.DTI_SSFP`; the `*_slide_deck_in_MRI` volumes and the registration FA (`dti_FA` for Vlad, `dtifit_FA` for Moe) from `PS-OCT/Enface/Registration`.
- `warp/`: `MRI_to_PSOCT_warpfield.nii.gz` (2.6–2.9 GB) and `Ret_slide_deck.nii.gz` from `PS-OCT/Enface/Registration`.

**These are pseudo-labels.** Agreement with them measures agreement with the MRI, not accuracy; 0.4 mm MRI voxels are coarse next to 55 µm PS-OCT pixels.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from scipy.ndimage import binary_erosion
from scipy.signal import fftconvolve
from sklearn.metrics import roc_auc_score

from brainseg import IGNORE, LABELS
from brainseg.mri_labels import label_mri, warp

DATA, OUT = Path("../data"), Path("../outputs")
SUBJECTS = {"vlad": {"mask": "brain_mask", "registration_fa": "dti_FA", "section_axis": 0, "through_axis": 0, "plane": "sagittal"},
            "moe": {"mask": "nodif_brain_mask", "registration_fa": "registration_dtifit_FA", "section_axis": 1, "through_axis": 1, "plane": "coronal"}}


def load(subject, name):
    return np.asarray(nib.load(DATA / subject / "mri" / f"{name}.nii.gz").dataobj, dtype=np.float32)

## Put the MRI and the registration grid together

The PS-OCT `*_in_MRI` volumes are on the grid of the FA the registration used. For Vlad that is the MRI grid. Moe's registration used `dtifit_FA`: the same image, scaled differently, on a padded 346 × 346 grid with a placeholder origin. Find Moe's offset by cross-correlation.

In [ ]:
for s, cfg in SUBJECTS.items():
    fa, reg = load(s, "dti_FA"), load(s, cfg["registration_fa"])
    cc = fftconvolve(reg, (fa - fa.mean())[::-1, ::-1, ::-1], mode="valid")
    cfg["offset"] = offset = tuple(int(i) for i in np.unravel_index(np.argmax(cc), cc.shape))
    crop = reg[tuple(slice(o, o + n) for o, n in zip(offset, fa.shape))]
    print(f"{s}: MRI {fa.shape} sits at offset {offset} in the registration grid {reg.shape}, corr {np.corrcoef(fa.ravel(), crop.ravel())[0, 1]:.3f}")

## Label the MRI

`label_mri` runs three-class Atropos on FA, T2 and log-MD, names the classes (fluid only if its median MD is clearly elevated; Moe has no such class), and marks partial volume: high-FA "fluid" deep in the brain (corpus callosum, white matter next to ventricles) becomes white matter, high-FA "fluid" on the surface and low-confidence voxels become 255.

In [ ]:
labels, psoct_in_mri = {}, {}
for s, cfg in SUBJECTS.items():
    fa_img = nib.load(DATA / s / "mri" / "dti_FA.nii.gz")
    fa, mask = load(s, "dti_FA"), load(s, cfg["mask"]) > 0
    labels[s] = label_mri(fa, load(s, "T2map"), load(s, "dti_MD"), mask, nib.affines.voxel_sizes(fa_img.affine))
    nib.save(nib.Nifti1Image(labels[s], fa_img.affine, fa_img.header), OUT / f"{s}_mri_labels.nii.gz")
    crop = tuple(slice(o, o + n) for o, n in zip(cfg["offset"], fa.shape))
    psoct_in_mri[s] = {c: load(s, f"{c}_slide_deck_in_MRI")[crop] for c in ("Ret", "Ref", "Cro")}
    shares = {LABELS.get(c, "uncertain"): f"{(labels[s] == c).sum() / mask.sum():.1%}" for c in (1, 2, 3, IGNORE)}
    print(s, shares)

## Check the labels against the PS-OCT (MRI space)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for ax, (s, cfg) in zip(axes, SUBJECTS.items()):
    lab, ret = labels[s], psoct_in_mri[s]["Ret"]
    classes = [c for c in (1, 2, 3) if (lab == c).any()]
    ax.boxplot([ret[lab == c] for c in classes], showfliers=False)
    ax.set(xticks=range(1, len(classes) + 1), xticklabels=[LABELS[c] for c in classes], title=f"{s}: PS-OCT retardance by MRI label")
    gw = (lab == 1) | (lab == 2)
    print(f"{s}: AUC, retardance separating MRI white from gray matter = {roc_auc_score(lab[gw] == 2, ret[gw]):.3f}")

Retardance measures birefringence in the section plane, so white matter whose fibers run through the section should look darker (the lab's "tissue facing a particular direction looks darker"). DTI's principal direction `dti_V1` tests this: the through-plane component is x for Vlad (sagittal) and y for Moe (coronal).

In [ ]:
for s, cfg in SUBJECTS.items():
    v1, wm = load(s, "dti_V1"), labels[s] == 2
    through = np.abs(v1[..., cfg["through_axis"]])[wm]
    ret = psoct_in_mri[s]["Ret"][wm]
    bins = [(0, 0.3), (0.3, 0.7), (0.7, 1.01)]
    medians = [f"{np.median(ret[(through >= lo) & (through < hi)]):.3g}" for lo, hi in bins]
    print(f"{s} ({cfg['plane']}): white-matter retardance median for in-plane / oblique / through-plane fibers: {' / '.join(medians)};"
          f" corr {np.corrcoef(ret, through)[0, 1]:.2f}")

## Warp into PS-OCT space

Nearest-neighbour resampling through the FNIRT warp (fslpy, no FSL install needed). Each warp needs ~25 GB of RAM and about 1.5 minutes; outputs are cached in `outputs/`. Moe's labels are first placed back into the padded registration grid.

In [ ]:
psoct_labels = {}
for s, cfg in SUBJECTS.items():
    path = OUT / f"{s}_labels_psoct.nii.gz"
    deck = nib.load(DATA / s / "warp" / "Ret_slide_deck.nii.gz")
    if not path.exists():
        source = nib.load(DATA / s / "mri" / f"{cfg['registration_fa']}.nii.gz")
        padded = np.zeros(source.shape, np.uint8)
        padded[tuple(slice(o, o + n) for o, n in zip(cfg["offset"], labels[s].shape))] = labels[s]
        out = warp(padded, source, DATA / s / "warp" / "MRI_to_PSOCT_warpfield.nii.gz", DATA / s / "warp" / "Ret_slide_deck.nii.gz")
        nib.save(nib.Nifti1Image(out, deck.affine), path)
    psoct_labels[s] = np.asarray(nib.load(path).dataobj)
    ret = np.asarray(deck.dataobj, dtype=np.float32)
    sample = np.flatnonzero((psoct_labels[s] == 1) | (psoct_labels[s] == 2))[::20]
    lab, r = psoct_labels[s].ravel()[sample], ret.ravel()[sample]
    print(f"{s}: PS-OCT grid {psoct_labels[s].shape}, AUC retardance white vs gray {roc_auc_score(lab == 2, r):.3f}, "
          f"labels on PS-OCT signal {((ret > 0) & (psoct_labels[s] > 0)).sum() / (psoct_labels[s] > 0).sum():.1%}")
    cfg["deck"] = ret

Overlays on slide-deck sections at full PS-OCT resolution: white matter in orange, fluid in blue, uncertain in red. The deck applies each section's in-plane transform (`abs_mat.json`), so these labels line up with the `*_slide_deck.nii.gz` volumes, not directly with the `Registration/<contrast>/lowres/` section files.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout="constrained")
for row, (s, cfg) in zip(axes, SUBJECTS.items()):
    mapping = {int(k): re.search(r"Slice_\d+", v).group(0) for k, v in json.load(open(DATA / s / "slidedeck_slice_mapping.json")).items()}
    n = cfg["deck"].shape[cfg["section_axis"]]
    for ax, i in zip(row, (n // 4, n // 2, 3 * n // 4)):
        img, lab = np.take(cfg["deck"], i, cfg["section_axis"]), np.take(psoct_labels[s], i, cfg["section_axis"])
        ax.imshow(np.rot90(img), cmap="gray", vmin=0, vmax=np.percentile(img[img > 0], 99))
        for c, color in ((2, "tab:orange"), (3, "tab:blue"), (IGNORE, "tab:red")):
            if (lab == c).any():
                ax.contour(np.rot90(lab == c), levels=[0.5], colors=color, linewidths=0.5)
        ax.set(xticks=[], yticks=[], title=f"{s}: deck {i} = {mapping[i]}")

## Findings

- **Registration holds.** Vlad's MRI FA and PS-OCT retardance correlate best at zero offset; Moe's MRI sits at a fixed voxel offset inside its padded registration grid.
- **Gray vs white matter: usable references for both brains.** PS-OCT retardance separates MRI white from gray matter (AUCs above), and still does after warping into PS-OCT space. In the overlays, Moe's white-matter contours follow the tissue closely; Vlad's are looser, especially near the midline and the surface, so treat Vlad's labels with more caution.
- **Fiber direction: a small effect.** White matter whose fibers run through the section has somewhat lower retardance (medians above), consistent with the lab's observation, but it is a few percent to ~10%, not a large bias.
- **Fluid: weak or absent.** Vlad has a small fluid class; Moe's MRI shows no free-water class at all. Fluid in sulci is below MRI resolution and needs hand labels.
- **Uncertain (255)** is mostly the brain surface shell, where 0.4 mm voxels mix tissue and medium.

To use these labels in the benchmark, take 2D sections from the `*_slide_deck.nii.gz` volumes (all four contrasts exist as decks), not from the separately registered section files.